In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
import pennylane as qml
from pennylane.qnn import TorchLayer

# ─────────────────────────────────────────────
# SEEDING
# ─────────────────────────────────────────────
def seed_all(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
# ✅ CHANGE 1: Increased n_qubits 4→8 for richer quantum expressivity
# ✅ CHANGE 2: Increased q_depth 4→6 for deeper variational circuit
n_qubits     = 8           # was 4 — 8 qubits × 3-axis measurement = 24 output features
q_depth      = 6           # was 4 — deeper circuit for better approximation
batch_size   = 32          # reduced from 64 — smaller batches act as regularizer
num_classes  = 10
num_epochs   = 80
lr           = 0.0005
weight_decay = 1e-4

# ─────────────────────────────────────────────
# TRANSFORMS  (no augmentation — VirusMNIST constraint)
# ─────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

eval_transform = transforms.Compose([
    transforms.Grayscale(1),
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# ─────────────────────────────────────────────
# DATASETS & LOADERS
# ─────────────────────────────────────────────
TRAIN_PATH = 'virus_MNIST dataset/train_balanced_v2'
TEST_PATH  = 'virus_MNIST dataset/test'
VAL_PATH   = 'virus_MNIST dataset/val'

try:
    train_dataset = ImageFolder(TRAIN_PATH, transform=train_transform)
    test_dataset  = ImageFolder(TEST_PATH,  transform=eval_transform)
    val_dataset   = ImageFolder(VAL_PATH,   transform=eval_transform)
    print("Datasets loaded successfully")
except Exception as e:
    print(f"Error loading datasets: {e}")

try:
    labels = [label for _, label in train_dataset.samples]
    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
    class_weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
    print("Class weights computed:", class_weights)
except Exception as e:
    print(f"Could not compute class weights: {e}. Using uniform weights.")
    class_weight_tensor = torch.ones(num_classes).to(device)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

# ─────────────────────────────────────────────
# QUANTUM CIRCUIT  — v2 with 3 key upgrades
# ─────────────────────────────────────────────
# UPGRADE A: Data Re-Uploading at every variational layer
#            → gives circuit universal approximation ability
#            → proven theoretically superior to single-encoding
#
# UPGRADE B: Brick-layer entanglement with trainable CRZ gates
#            → alternating even/odd qubit pairs per layer
#            → CRZ learns *how much* to entangle (vs fixed CNOT)
#
# UPGRADE C: 3-axis Pauli measurement (X, Y, Z per qubit)
#            → triples output features: 8 qubits × 3 = 24 values
#            → captures full Bloch sphere information, not just Z-axis
# ─────────────────────────────────────────────
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def quantum_circuit(inputs, weights):
    # ── Initial Angle Encoding ───────────────────────────────────
    # Each qubit receives RY + RZ rotations from bridge output
    # inputs shape: (batch, 2 * n_qubits)
    for i in range(n_qubits):
        qml.RY(inputs[..., i],            wires=i)
        qml.RZ(inputs[..., i + n_qubits], wires=i)

    # ── Variational Layers with Data Re-Uploading ────────────────
    for l in range(weights.shape[0]):

        # UPGRADE B: Brick-layer entanglement with trainable CRZ
        # Even layers: entangle (0,1), (2,3), (4,5), (6,7)
        # Odd  layers: entangle (1,2), (3,4), (5,6)
        if l % 2 == 0:
            for i in range(0, n_qubits - 1, 2):
                qml.CRZ(weights[l, i, 2], wires=[i, i + 1])
        else:
            for i in range(1, n_qubits - 1, 2):
                qml.CRZ(weights[l, i, 2], wires=[i, i + 1])

        # UPGRADE A: Re-upload input data at EVERY layer
        # Combines trainable weights with input angles → data-dependent rotations
        for i in range(n_qubits):
            qml.RY(weights[l, i, 0] + inputs[..., i],            wires=i)
            qml.RZ(weights[l, i, 1] + inputs[..., i + n_qubits], wires=i)

    # UPGRADE C: Measure all 3 Pauli axes per qubit = 3 × n_qubits features
    measurements = []
    for i in range(n_qubits):
        measurements.append(qml.expval(qml.PauliZ(i)))
        measurements.append(qml.expval(qml.PauliX(i)))
        measurements.append(qml.expval(qml.PauliY(i)))
    return measurements

# shape: (q_depth, n_qubits, 3)  ← 3rd dim now includes CRZ weight
weight_shapes = {"weights": (q_depth, n_qubits, 3)}

# Output dimension of quantum layer = 3 * n_qubits
q_out_dim = 3 * n_qubits   # = 24 features for n_qubits=8

# ─────────────────────────────────────────────
# FOCAL LOSS  — addresses Class 0 F1=0.38 problem
# ─────────────────────────────────────────────
# Standard CrossEntropy ignores hard examples once loss is low.
# Focal Loss (γ=2) penalises misclassified hard samples more,
# forcing the model to focus on Class 0 and other weak classes.
# ─────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.weight          = weight
        self.gamma           = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss    = F.cross_entropy(
            inputs, targets,
            weight=self.weight,
            label_smoothing=self.label_smoothing,
            reduction='none'
        )
        pt         = torch.exp(-ce_loss)                   # probability of correct class
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss    # down-weight easy examples
        return focal_loss.mean()


# ─────────────────────────────────────────────
# BUILDING BLOCKS
# ─────────────────────────────────────────────
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        scale = self.pool(x).view(b, c)
        scale = self.fc(scale).view(b, c, 1, 1)
        return x * scale


# ✅ CHANGE 3: Added stochastic depth (DropPath) to ResBlock
# DropPath randomly drops entire residual branches during training
# — strong regulariser that reduces the train/val accuracy gap
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, dropout=0.15, drop_path=0.1):
        super().__init__()
        # ✅ CHANGE 4: dropout 0.1→0.15 for stronger regularisation
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_ch)
        )
        self.se            = SEBlock(out_ch)
        self.drop_path_rate = drop_path
        self.skip = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm2d(out_ch)
        ) if (stride != 1 or in_ch != out_ch) else nn.Identity()
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.conv_block(x)
        out = self.se(out)
        # Stochastic depth: randomly zero out entire residual path per sample
        if self.training and self.drop_path_rate > 0:
            keep_prob = 1 - self.drop_path_rate
            # Shape: (batch, 1, 1, 1) — drop entire feature maps, not individual values
            random_tensor = torch.rand(
                x.shape[0], 1, 1, 1, device=x.device
            ) < keep_prob
            out = out * random_tensor.float() / keep_prob
        return self.relu(out + self.skip(x))


# ─────────────────────────────────────────────
# QUANTUM BRIDGE v2  — learnable angle scaling
# ─────────────────────────────────────────────
# UPGRADE D: Learnable per-angle scale + bias parameters
# The original tanh(x)*π saturates quickly, clustering most
# angles near 0. Sigmoid + learned scale/bias spreads the
# distribution more evenly across the full rotation range,
# giving each qubit/axis a distinct, meaningful input angle.
# ─────────────────────────────────────────────
class QuantumBridge(nn.Module):
    def __init__(self, in_features, n_qubits):
        super().__init__()
        self.project = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(0.2),                   # ← added bridge dropout
            nn.Linear(64, n_qubits * 2)
        )
        # Learnable scale and bias — let the model discover optimal angle ranges
        self.angle_scale = nn.Parameter(torch.ones(n_qubits * 2) * torch.pi)
        self.angle_bias  = nn.Parameter(torch.zeros(n_qubits * 2))

    def forward(self, x):
        x = self.project(x)
        # sigmoid maps to (0,1), then scale to learned range
        # prevents saturation clustering near 0 that tanh caused
        return self.angle_scale * torch.sigmoid(x) + self.angle_bias


# ─────────────────────────────────────────────
# MAIN MODEL  —  HybridResNet v2
# ─────────────────────────────────────────────
#
#  Flow (unchanged — all data passes through quantum layer):
#  Input(1,32,32)
#    → Stem           (1→32)
#    → Stage1         (32→32, no downsample)       [+stochastic depth]
#    → Stage2         (32→64, stride=2)            [+stochastic depth]
#    → Stage3         (64→128, stride=2)           [+stochastic depth]
#    → GAP            (128,8,8 → 128)
#    → QuantumBridge  (128 → 2*n_qubits angles)   [+learnable scale]
#    → QuantumLayer   (angles → 3*n_qubits values) [+re-upload+CRZ+3axis]
#    → Classifier     (24 → num_classes)           [simplified head]
#
class HybridResNet(nn.Module):
    def __init__(self, n_qubits, q_out_dim, num_classes, dropout=0.3):
        super().__init__()

        # ── Classical Backbone ────────────────────────────────────
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        # Progressive stochastic depth rates (deeper = higher drop rate)
        self.stage1 = nn.Sequential(
            ResBlock(32, 32,  drop_path=0.05),
            ResBlock(32, 32,  drop_path=0.05)
        )
        self.stage2 = nn.Sequential(
            ResBlock(32,  64, stride=2, drop_path=0.10),
            ResBlock(64,  64,           drop_path=0.10)
        )
        self.stage3 = nn.Sequential(
            ResBlock(64,  128, stride=2, drop_path=0.15),
            ResBlock(128, 128,           drop_path=0.15)
        )
        self.gap = nn.AdaptiveAvgPool2d(1)

        # ── Quantum Bridge (v2) ───────────────────────────────────
        self.bridge = QuantumBridge(in_features=128, n_qubits=n_qubits)

        # ── Quantum Layer ─────────────────────────────────────────
        self.q_layer = TorchLayer(quantum_circuit, weight_shapes)

        # ── Classifier Head (v2) — proportional to q_out_dim ──────
        # ✅ CHANGE 5: Simplified head — 2 layers instead of 3
        # Large heads overfit the small quantum output; keep it proportional
        self.classifier = nn.Sequential(
            nn.Linear(q_out_dim, q_out_dim * 2),   # 24→48
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(q_out_dim * 2, num_classes)   # 48→10
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        # Classical feature extraction
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)    # (B, 128)

        # Bridge: 128 → quantum angle encodings
        x = self.bridge(x)            # (B, 2*n_qubits)

        # Quantum processing with re-uploading + 3-axis measurement
        x = self.q_layer(x)           # (B, 3*n_qubits) = (B, 24)

        # Classification
        return self.classifier(x)


# ─────────────────────────────────────────────
# TRAINING & EVALUATION
# ─────────────────────────────────────────────
def train_epoch(model, dataloader, loss_fn, optimizer, scheduler, device):
    model.train()
    total_loss, correct = 0.0, 0

    for inputs, labels in tqdm(dataloader, desc="Training", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = loss_fn(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()   # ✅ OneCycleLR must step every BATCH, not every epoch

        total_loss += loss.item()
        correct    += (outputs.argmax(dim=1) == labels).sum().item()

    return total_loss / len(dataloader), correct / len(dataloader.dataset)


def evaluate(model, dataloader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs        = model(inputs)
            loss           = loss_fn(outputs, labels)
            total_loss    += loss.item()
            _, predicted   = torch.max(outputs, 1)
            total         += labels.size(0)
            correct       += (predicted == labels).sum().item()

    return total_loss / len(dataloader), correct / total


# ─────────────────────────────────────────────
# MODEL, OPTIMIZER, LOSS, SCHEDULER
# ─────────────────────────────────────────────
model = HybridResNet(
    n_qubits    = n_qubits,
    q_out_dim   = q_out_dim,
    num_classes = num_classes,
    dropout     = 0.3
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

# ✅ CHANGE 6: Separate LRs preserved; quantum layer still 10× slower
optimizer = torch.optim.AdamW([
    {'params': model.stem.parameters(),       'lr': lr},
    {'params': model.stage1.parameters(),     'lr': lr},
    {'params': model.stage2.parameters(),     'lr': lr},
    {'params': model.stage3.parameters(),     'lr': lr},
    {'params': model.bridge.parameters(),     'lr': lr},
    {'params': model.q_layer.parameters(),    'lr': lr * 0.1},   # quantum needs slower LR
    {'params': model.classifier.parameters(), 'lr': lr},
], weight_decay=weight_decay)

# ✅ CHANGE 7: Focal Loss replaces CrossEntropy
# gamma=2.0 focuses training on hard/misclassified examples (esp. Class 0)
loss_fn = FocalLoss(weight=class_weight_tensor, gamma=2.0, label_smoothing=0.1)

# ✅ CHANGE 8: OneCycleLR — faster convergence and better generalization
# pct_start=0.3 → spend 30% of training warming up, then anneal
# Separate max_lr per param group matches the AdamW group order
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr        = [lr, lr, lr, lr, lr, lr * 0.1, lr],
    steps_per_epoch = len(train_loader),
    epochs        = num_epochs,
    pct_start     = 0.3,
    anneal_strategy = 'cos',
    div_factor    = 10.0,    # initial_lr = max_lr / 10
    final_div_factor = 1e4   # min_lr = max_lr / 10000
)

# ─────────────────────────────────────────────
# TRAINING LOOP
# ─────────────────────────────────────────────
best_val_acc               = 0.0
train_losses, val_losses   = [], []
train_accs,   val_accs     = [], []
early_stopping_patience    = 15
epochs_without_improvement = 0

print(f"\nStarting Hybrid v2 Training for {num_epochs} epochs...")
print(f"Quantum config: {n_qubits} qubits | {q_depth} layers | {q_out_dim} output features")
print("=" * 60)

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, loss_fn, optimizer, scheduler, device)
    val_loss,   val_acc   = evaluate(model, val_loader, loss_fn, device)
    # ✅ No scheduler.step() here — OneCycleLR already steps per batch inside train_epoch

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch [{epoch+1:02d}/{num_epochs}] | LR: {current_lr:.6f}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"  Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc               = val_acc
        epochs_without_improvement = 0
        torch.save({
            'epoch':                epoch + 1,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc':              val_acc,
            'val_loss':             val_loss,
            'config': {
                'n_qubits':   n_qubits,
                'q_depth':    q_depth,
                'q_out_dim':  q_out_dim,
                'num_classes': num_classes,
            }
        }, "after_RESNET4.pth")
        print(f"  💾 Best model saved (Val Acc: {best_val_acc:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"  🕒 No improvement for {epochs_without_improvement} epoch(s).")

    if epochs_without_improvement >= early_stopping_patience:
        print(f"\n⏹️  Early stopping triggered after {epoch+1} epochs.")
        break

    print("-" * 60)

print(f"\n✅ Training complete. Best Val Acc: {best_val_acc:.4f}")

Using device: cuda
Datasets loaded successfully
Class weights computed: [1.64455748 0.74958269 1.35924888 1.72713675 5.20923738 0.86820623
 0.37487826 0.76944312 1.60483124 1.23593272]
Total trainable parameters: 716,794

Starting Hybrid v2 Training for 80 epochs...
Quantum config: 8 qubits | 6 layers | 24 output features


Epoch [01/80] | LR: 0.000052
  Train Loss: 1.7678 | Train Acc: 0.2369
  Val   Loss: 1.1520 | Val   Acc: 0.4594
  💾 Best model saved (Val Acc: 0.4594)
------------------------------------------------------------


Epoch [02/80] | LR: 0.000058
  Train Loss: 1.3389 | Train Acc: 0.4213
  Val   Loss: 0.8092 | Val   Acc: 0.5998
  💾 Best model saved (Val Acc: 0.5998)
------------------------------------------------------------


Epoch [03/80] | LR: 0.000067
  Train Loss: 1.0337 | Train Acc: 0.5556
  Val   Loss: 0.6078 | Val   Acc: 0.7164
  💾 Best model saved (Val Acc: 0.7164)
------------------------------------------------------------


Epoch [04/80] | LR: 0.000080
  Train Loss: 0.8527 | Train Acc: 0.6350
  Val   Loss: 0.5349 | Val   Acc: 0.7916
  💾 Best model saved (Val Acc: 0.7916)
------------------------------------------------------------


...

Epoch [09/80] | LR: 0.000189
  Train Loss: 0.5549 | Train Acc: 0.8316
  Val   Loss: 0.4123 | Val   Acc: 0.8846
  💾 Best model saved (Val Acc: 0.8846)
------------------------------------------------------------


Epoch [10/80] | LR: 0.000217
  Train Loss: 0.5324 | Train Acc: 0.8433
  Val   Loss: 0.4014 | Val   Acc: 0.8944
  💾 Best model saved (Val Acc: 0.8944)
------------------------------------------------------------


Epoch [11/80] | LR: 0.000246
  Train Loss: 0.5035 | Train Acc: 0.8582
  Val   Loss: 0.4028 | Val   Acc: 0.8706
  🕒 No improvement for 1 epoch(s).
------------------------------------------------------------


Epoch [12/80] | LR: 0.000275
  Train Loss: 0.4919 | Train Acc: 0.8653
  Val   Loss: 0.3914 | Val   Acc: 0.8865
  🕒 No improvement for 2 epoch(s).
------------------------------------------------------------


Training:  17%|█▋        | 253/1516 [00:41<03:15,  6.46it/s]